[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gfascioli/NeuralNetwork_SMS_TextClassifier_freeCodeCamp_MachineLearningWithPython/blob/main/fcc_sms_text_classification.ipynb)

# 💬 Neural Network SMS Text Classifier

### freeCodeCamp — Machine Learning with Python Certification

**Goal:** classify SMS messages as `ham` (a normal message) or `spam` (an ad or unsolicited message), via a function `predict_message(text)` that returns `[likelihood_of_spam, "ham" or "spam"]`.

The plan:
1. Load the SMS Spam Collection train/test TSV files.
2. Map labels to 0 (ham) / 1 (spam).
3. Build a model with text vectorization built in, so it can take raw strings directly.
4. Train it, correcting for the dataset's natural ham/spam imbalance.
5. Define `predict_message` and confirm it against the official test cell.

---
*Solution by **Gonzalo Fascioli** — [github.com/Gfascioli](https://github.com/Gfascioli)*

In [ ]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(tf.__version__)


In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"


Write all your code in the cell below — the final cell is the official grader and should stay exactly as provided.

In [ ]:
# --- Solution by Gonzalo Fascioli (github.com/Gfascioli) ---

# each file has two tab-separated columns: label ("ham"/"spam"), then the message
train_dataset = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'message'])
test_dataset = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'message'])

train_dataset['label'] = train_dataset['label'].map({'ham': 0, 'spam': 1})
test_dataset['label'] = test_dataset['label'].map({'ham': 0, 'spam': 1})

train_messages = train_dataset['message'].values
train_labels = train_dataset['label'].values
test_messages = test_dataset['message'].values
test_labels = test_dataset['label'].values

# Text vectorization lives INSIDE the model (as its first layer), so
# predict_message can hand the model a raw string directly -- no separate
# tokenizing/padding step to keep in sync at prediction time.
VOCAB_SIZE = 2000
MAX_LEN = 60

vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)
vectorize_layer.adapt(train_messages)

model = keras.Sequential([
    vectorize_layer,
    layers.Embedding(VOCAB_SIZE, 16, mask_zero=True),
    layers.GlobalAveragePooling1D(),
    layers.Dense(24, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# the dataset is imbalanced (far more "ham" than "spam"), so weight the
# minority class higher to stop the model from just guessing "ham" every time
neg, pos = np.bincount(train_labels)
total = neg + pos
class_weight = {0: total / (2.0 * neg), 1: total / (2.0 * pos)}

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    train_messages,
    train_labels,
    epochs=30,
    validation_data=(test_messages, test_labels),
    class_weight=class_weight,
    verbose=0,
    callbacks=[early_stop]
)

print("Training finished after {} epochs.".format(len(history.history['loss'])))

def predict_message(pred_text):
    # NOTE: model.predict() on a single raw-string input hits a Keras 3
    # data-adapter bug ("Invalid dtype: strNNN"), so we call the model
    # directly instead -- that runs the layers' native TensorFlow ops
    # without going through the buggy generic adapter. training=False
    # turns off Dropout, matching what .predict() would normally do.
    input_tensor = tf.constant([pred_text], dtype=tf.string)
    prediction = model(input_tensor, training=False)
    prob = float(prediction[0][0])
    label = 'spam' if prob >= 0.5 else 'ham'
    return [prob, label]


Run the cell below to test your model and function.

**This is the official freeCodeCamp grading cell — leave it unmodified.**

In [ ]:
# Run this cell to test your model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                    "sale today! to stop texts call 98912460324",
                    "i dont want to go. can we try it a different day? available sat",
                    "our new mobile video service is live. just install on your phone to start watching.",
                    "you have won £1000 cash! call to claim your prize.",
                    "i'll bring it tomorrow. don't forget the milk.",
                    "wow, is your arm alright. that happened to me one time too"
                    ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed. Keep trying.")

test_predictions()
